In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def random_gate(p, q, r):
    # Define the weights for the different operations based on the given probabilities
    weights = [1 - p - q - r, p, q, r]
    ops = [qml.Identity, qml.PauliX, qml.PauliY, qml.PauliZ]
    
    # Initialize the density matrix for the state |0>
    density_matrix = np.array([[1, 0], [0, 0]], dtype=complex)
    
    # Apply the mixed state formulation
    mixed_density_matrix = sum([w * qml.apply(op, density_matrix) for w, op in zip(weights, ops)])
    
    # Compute the probabilities from the diagonal of the mixed density matrix
    probs = np.real(np.diag(mixed_density_matrix))
    return probs

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = random_gate(*ins).numpy().tolist()
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol = 1e-4), "Not the correct probabilities"

test_cases = [
    ('[0.25, 0.25, 0.25]', '[0.5, 0.5]'),
    ('[0.125, 0.25, 0.2]', '[0.625, 0.375]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError as e:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'. {e}")


Running test case 0 with input '[0.25, 0.25, 0.25]'...
Runtime Error. A quantum function must return either a single measurement, or a nonempty sequence of measurements.
Running test case 1 with input '[0.125, 0.25, 0.2]'...
Runtime Error. A quantum function must return either a single measurement, or a nonempty sequence of measurements.
